In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Khai báo thư viện

In [ ]:
!pip install transformers==4.52.4
!pip install bitsandbytes==0.46.0
!pip install accelerate==1.7.0
!pip install langchain==0.3.25
!pip install langchainhub==0.1.21
!pip install langchain-chroma==0.2.4
!pip install langchain_experimental==0.3.4
!pip install langchain-community==0.3.24
!pip install langchain_huggingface==0.2.0
!pip install python-dotenv==1.1.0
!pip install pypdf

In [ ]:
import torch
from transformers import BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain.memory import ConversationBufferMemory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.chains import ConversationalRetrievalChain
from langchain_experimental.text_splitter import SemanticChunker
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain import hub

# Load data + xử lý dữ liệu đầu vào -> embedding

In [ ]:
Loader = PyPDFLoader
FILE_PATH = "/content/drive/MyDrive/AIO/code-data/YOLOv10_Tutorials.pdf"
loader = Loader(FILE_PATH)
documents = loader.load()

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="bkai-foundation-models/vietnamese-bi-encoder")

In [ ]:
# Chia document thành các chunk dựa trên ý nghĩa (semantic)
# -> không phải cắt theo số ký tự như bình thường
semantic_splitter = SemanticChunker(
    embeddings=embedding,
    # Vai trò:
    # Biến từng câu → vector
    # So sánh độ giống nhau giữa các câu
    # SemanticChunker dùng nó để: xác định chỗ nào nội dung bị “đổi ý” -> cắt chunk

    buffer_size=1,
    # Ý nghĩa:
    # Khi xét 1 câu, nó sẽ nhìn thêm các câu xung quanh
    # buffer = 1 nghĩa là:
    # mỗi lần xét:
    # câu trước + câu hiện tại + câu sau

    breakpoint_threshold_type="percentile",
    # Đây là cách xác định khi nào thì cắt đoạn
    # "percentile" nghĩa là:
    # không dùng ngưỡng cố định
    # mà dùng phân vị (percentile)
    # -> tìm những điểm “khác biệt ngữ nghĩa mạnh nhất”

    breakpoint_threshold_amount=95,
    # Kết hợp với dòng trên:
    # Nghĩa là:
    # chỉ cắt ở những điểm: độ tương đồng thấp hơn 95%

    # breakpoint_threshold_type="percentile"
    # breakpoint_threshold_amount=95
    # -> tóm gọn chỉ cắt ở những phân vị (percentile) có độ tương đồng thấp hơn 95%
    # -> cắt ở chỗ ngữ nghĩa thay đổi mạnh nhất "toàn bộ văn bản"

    min_chunk_size=500, # tối thiểu = 500 ký tự
    add_start_index=True # biết chunk này bắt đầu từ vị trí nào trong document

)

- Văn bản được chia thành các **"câu"**.
- Sau đó các câu được nhóm lại thành các window, mỗi window = 3 câu **(buffer=1)**.
- Hệ thống tính độ khác biệt ngữ nghĩa giữa các window liên tiếp.
- Dựa vào các giá trị này, nó chọn ra những điểm có sự thay đổi mạnh nhất để cắt

-> **"Các chunk"**: là tập hợp của nhiều câu liên tiếp, không còn là từng câu riêng lẻ.

**Kết quả sau khi gọi hàm .split_documents ở code bên dưới** thì sẽ tạo thành các chunk chứa các câu (đây là bước embedding để tạo thành chunk) chứ bước embed này không tạo thành kết quả và lưu trữ vector

In [ ]:
docs = semantic_splitter.split_documents(documents)
print("Number of semantic chunks: ", len(docs))

In [ ]:
# Bước này mới là bước embedding thành vector
vector_db = Chroma.from_documents(documents=docs,
                                  embedding=embedding)
retriever = vector_db.as_retriever()

# mỗi chunk → 1 vector
# lưu vào vector DB
# phục vụ truy vấn (retrieval)

Ta sử dụng Chroma để khởi
tạo vector database từ các documents đã được chia thành chunk
và được vector hóa.

## Check xem retrieval có hoạt động không

In [ ]:
result = retriever.invoke("What is YOLO?")

print("Number of relevant documents: ", len(result))

# Xây dựng model + pipeline

In [ ]:
nf4_config = BitsAndBytesConfig(
  load_in_4bit=True,    # load model ở dạng 4 bit
  bnb_4bit_quant_type="nf4",  # NF4 = Normal Float 4
  # cách nén số thực tối ưu cho LLM
  bnb_4bit_use_double_quant=True, # Double quantization -> nén thêm 1 lần nữa
  bnb_4bit_compute_dtype=torch.bfloat16  # Khi tính toán (inference) → dùng bfloat16
)

In [ ]:
MODEL_NAME = "lmsys/vicuna-7b-v1.5"
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME,
  quantization_config=nf4_config,
  low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
model_pipeline = pipeline(
    "text-generation",  # "text-generation" = sinh văn bản (LLM kiểu GPT)
    # Nếu đổi:
    # "summarization" -> tóm tắt
    # "translation" -> dịch
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512, # Giới hạn số token model được sinh thêm
    pad_token_id=tokenizer.eos_token_id, # Dùng token kết thúc câu (EOS) để làm padding
    device_map="auto"
)

llm = HuggingFacePipeline(
    pipeline=model_pipeline,
)

## Dùng prompt + chạy model trả về output -> đổi thành string

In [ ]:
# HuggingFacePipeline là wrapper của LangChain
# Mục đích là biến model_pipeline → thành LLM chuẩn LangChain
# -> LangChain không làm việc trực tiếp với Hugging Face pipeline -> cần một interface chung gọi là LLM
# nên phải "wrap" lại

prompt = hub.pull("rlm/rag-prompt") # Lấy prompt từ LangChain Hub

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
  {"context": retriever | format_docs, "question":
  RunnablePassthrough()}
  | prompt
  | llm
  | StrOutputParser() # Convert output về string
)

USER_QUESTION =  "YOLOv10 là gì?"
output = rag_chain.invoke(USER_QUESTION)
answer = output.split("Answer:")[1].strip()
print(answer)

Trước đó retriver được tạo bởi

- docs = semantic_splitter.split_documents(documents)
- vector_db = Chroma.from_documents(documents=docs, embedding=st.session_state.embeddings)
- retriever = vector_db.as_retriever()

-> Thế nên retriver đã có sẵn thiết kế của langchain bên trong -> khi dùng dấu **" | "** => tự hiểu rằng chạy từng thành phần bên trong

flow: khi đưa question vào rag_chain:
- Đưa question xyz vào rag_chain -> question đó sẽ được xử lý theo 2 cách
- Nhánh thứ 1 là retriver.invoke(question) sau đó là format_docs(question)
- Nhánh thứ 2: question được giữ nguyên không thay đổi
- Sau khi được xử lý trong 2 cặp keyvalue sẽ sinh ra data như sau: **{'context': Yolo là..., 'question': Yolo là gì?}**

-> ***conext*** chính là sau khi xử lý question và tìm ra câu trả lời tương ứng, còn ***question*** là câu hỏi được giữa nguyên
- Sau đó các hàm khác của langchain lại tiếp tục được chạy, kết quả có được trước đó là 2 cặp key-value là context và question sẽ được đưa vào prompt: ***promt.invoke(2 cặp key-value)*** đặt kết quả này là **x**
- Tiếp đến ***llm.invoke(x)*** đặt là **y**
- Tiếp đến là ***StrOutputParser(y)***